<a href="https://colab.research.google.com/github/joohyosung/KMU_AIstudy/blob/main/week08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai "pydantic>=2,<3"


In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
)


In [3]:
model = "google/gemini-2.5-flash-lite"


In [4]:
{
  "text": "생성형 AI 수업 너무 재밌어요.",
  "sentiment": "긍정",
  "keywords": ["재밌어요"]
}


{'text': '생성형 AI 수업 너무 재밌어요.', 'sentiment': '긍정', 'keywords': ['재밌어요']}

In [5]:
review = "생성형 AI 수업 너무 재밌어요."
messages = [
    {
        "role": "system",
        "content": (
            "문장의 감성을 분석해 JSON 객체로만 답해라. "
            "text에는 원문, sentiment에는 긍정·부정·중립 중 하나, "
            "keywords에는 감정을 나타내는 표현의 목록을 넣어라."
        ),
    },
    {"role": "user", "content": review},
]

json_response = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={"type": "json_object"},  # JSON 모드. 프롬프트에도 JSON 출력 명시
    max_tokens=500,  # 짧은 감성 분석 결과의 생성 토큰 상한
    # JSON 모드를 지원하는 서버에만 요청
    extra_body={"provider": {"require_parameters": True}},
)


In [6]:
json_response.choices[0].finish_reason  # 응답 생성이 끝난 이유 확인


'stop'

In [7]:
import json

json.loads(json_response.choices[0].message.content)  # JSON 문자열을 Python 값으로 변환


{'text': '생성형 AI 수업 너무 재밌어요.',
 'sentiment': 'positive',
 'keywords': ['너무 재밌어요']}

In [8]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field


class SentimentAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")  # 정의하지 않은 필드 금지
    text: str  # 필드명은 text, 자료형은 문자열(str)
    sentiment: Literal["긍정", "부정", "중립"]  # sentiment는 긍정, 부정, 중립 중 하나
    keywords: list[str] = Field(description="원문에서 감정을 나타내는 표현 목록")
    # keywords는 문자열 리스트. description은 설명

sentiment_schema = SentimentAnalysis.model_json_schema()  # 정의한 구조를 JSON Schema로
sentiment_schema["required"]  # 필수 필드 목록


['text', 'sentiment', 'keywords']

In [13]:
sample_sentiment = SentimentAnalysis.model_validate_json(
    '{"text": "생성형 AI 수업 너무 재밌어요.", '
    '"sentiment": "긍정", "keywords": ["재밌어요"]}'
)
sample_sentiment.model_dump()


{'text': '생성형 AI 수업 너무 재밌어요.', 'sentiment': '긍정', 'keywords': ['재밌어요']}

In [14]:
schema_response = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={
        "type": "json_schema",  # JSON Schema로 응답 구조 지정
        "json_schema": {
            "name": "sentiment_analysis",  # 스키마 이름
            "strict": True,  # 정의한 필드·자료형·허용 값에 맞춰 생성
            "schema": sentiment_schema,  # BaseModel에서 만든 JSON Schema 전달
        },
    },
    max_tokens=500,
    # JSON Schema를 지원하는 서버에만 요청
    extra_body={"provider": {"require_parameters": True}},
)


In [15]:
schema_response.choices[0].finish_reason


'stop'

In [16]:
sentiment_result = SentimentAnalysis.model_validate_json(
    schema_response.choices[0].message.content  # 모델이 생성한 JSON 문자열
)
sentiment_result.model_dump()  # 검증한 값을 Python 딕셔너리로 변환


{'text': '생성형 AI 수업 너무 재밌어요.', 'sentiment': '긍정', 'keywords': ['너무 재밌어요']}